# Compact MI Models
Locked target-only benchmark of installed Braindecode 1.4 compact models.

# 1. Setup

In [ ]:
from __future__ import annotations
import builtins, hashlib, json, os, platform, random, sys, time
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon
from sklearn.metrics import confusion_matrix
from modern_mi_common import *
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | CWD: {Path.cwd()}")

# 2. Configuration
## 2.1 Liu2024 Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-compact-mi-models"), "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"), "experiment_name": "compact_mi_locked_fbcnet", "config_note": "Locked target-only compact MI benchmark.",
    # Dataset and trial-independent preprocessing
    "subjects_to_use": None, "channel_set": "liu29", "native_sfreq": 500, "target_sfreq": 128, "marker_channel_index": 32, "onset_marker_value": 2, "onset_plausible_range": [800,1300], "window_seconds": 4.0, "bandpass_hz": [4.0,40.0], "filter_order": 4, "normalization_mode": "channel_standardize", "normalization_eps": 1e-6,
    # Model / evaluation
    "model_name": "FBCNet", "allow_optional_models": False, "evaluation_mode": "stratified_5fold", "cv_folds": 5, "cv_random_state": 2026, "persist_splits": True, "epoch_selection": "fixed_source_locked", "source_locked_epochs": 20,
    # Training / reproducibility / diagnostics
    "batch_size": 8, "n_epochs": 20, "learning_rate": 0.0003, "weight_decay": 0.01, "gradient_clip_norm": 1.0, "seeds": [2026], "seed": 2026, "set_seed": True, "bootstrap_iterations": 10000, "collapse_threshold": 0.9, "classical_reference_artifact": str(WORKING_DIR / "artifacts" / "liu2024-multiscale-riemann-fusion" / "20260712_145032_844360_89d4f1fc" / "subject_metrics.json"), "classical_reference_method": "riemann_equal_short_scales"
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id(); ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"; _LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"; stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep=kwargs.pop("sep"," "); end=kwargs.pop("end","\n"); flush=kwargs.pop("flush",False); file=kwargs.pop("file",None)
    message=sep.join(str(a) for a in args); target=sys.stdout if file is None else file; stamped=f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}"
    _safe_write_text(target, stamped+end); _safe_write_text(_LOG_FILE_HANDLE, stamped+end)
    if flush: target.flush(); _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path=ARTIFACT_DIR/"config.json"; config_path.write_text(json.dumps(CONFIG,indent=2),encoding="utf-8")
print(f"Run ID:     {RUN_ID}"); print(f"Artifacts:  {ARTIFACT_DIR}"); print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built(): return torch.device("mps")
    if torch.cuda.is_available(): return torch.device("cuda")
    return torch.device("cpu")
DEVICE=resolve_device(); print(f"Using device: {DEVICE}")
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"]=str(seed); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True,warn_only=True)
BASE_SEED=int(CONFIG["seed"])
if CONFIG["set_seed"]: seed_everything(BASE_SEED); print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Data Loading Helpers
Helpers are imported from `modern_mi_common.py`.
## 3.2 Trial-Independent Preprocessing
Full 8-s trials are independently filtered before marker-relative cropping.
## 3.3 Dataset Classes
TensorDataset is constructed fold-locally.
## 3.4 Locate and Load Data

# 4. Model
## 4.1 Build Model and Trainable Parameter Phases
The registry validates a real forward pass and fails loudly.
## 4.2 Parameter Diagnostics

# 5. Training
## 5.1 Build Classifier
## 5.2 Within-Subject Cross-Validation Runner
## 5.3 Run All Subjects

In [ ]:
print("Full CONFIG banner:\n"+json.dumps(CONFIG,indent=2))
if CONFIG["model_name"] in OPTIONAL_MODELS and not CONFIG["allow_optional_models"]: raise ValueError("Optional model requires allow_optional_models=true")
paths=locate_subject_files(CONFIG["source_extract_dir"],CONFIG["subjects_to_use"]); SUBJECTS=[subject_id(p) for p in paths]; FOLD_RESULTS=[]; inventory=[]; CH_NAMES=[]
for p in paths:
    x,y,CH_NAMES,onsets=load_subject(p,CONFIG); splits=make_splits(y,CONFIG,subject_id(p)); inventory.append({"subject_id":subject_id(p),"path":str(p),"n_trials":len(y),"split_hash":stable_hash(splits)})
    for split in splits:
        tr=np.asarray(split["train_indices"]); te=np.asarray(split["test_indices"]); mean,scale=fit_normalizer(x[tr],CONFIG["normalization_mode"],CONFIG["normalization_eps"]); seed_rows=[]
        for seed in CONFIG["seeds"]:
            seed_everything(seed); model=build_model(CONFIG["model_name"],x.shape[1],x.shape[2],CONFIG["target_sfreq"],DEVICE); pred,prob,elapsed=train_eval(model,((x[tr]-mean)/scale).astype('float32'),y[tr],((x[te]-mean)/scale).astype('float32'),y[te],CONFIG,seed,epochs=CONFIG["source_locked_epochs"]); seed_rows.append((pred,prob,elapsed,model))
        prob=np.mean([r[1] for r in seed_rows],axis=0); pred=prob.argmax(1); FOLD_RESULTS.append(fold_result(subject_id(p),split["fold_id"],te,y[te],pred,prob,seed_rows[-1][3],sum(r[2] for r in seed_rows),BASE_SEED,{"seed_ensemble":CONFIG["seeds"],"split_hash":stable_hash(split)}))
subject_inventory_path=ARTIFACT_DIR/"subject_inventory.csv"; pd.DataFrame(inventory).to_csv(subject_inventory_path,index=False)
(ARTIFACT_DIR/"splits.json").write_text(json.dumps({r["subject_id"]:make_splits(load_subject(Path(r["path"]),CONFIG)[1],CONFIG,r["subject_id"]) for r in inventory},indent=2),encoding="utf-8")

# 6. Results
## 6.1 Aggregate Metrics

In [ ]:
def aggregate_results(rows):
    subject_rows=[]
    for sid in sorted({r["subject_id"] for r in rows}):
        rr=[r for r in rows if r["subject_id"]==sid]; yt=np.concatenate([np.asarray(r["true_labels"]) for r in rr]); yp=np.concatenate([np.asarray(r["predictions"]) for r in rr])
        subject_rows.append({"subject_id":sid,"balanced_accuracy":float(__import__('sklearn').metrics.balanced_accuracy_score(yt,yp)),"accuracy":float((yt==yp).mean()),"n_trials":len(yt)})
    vals=[r["balanced_accuracy"] for r in subject_rows]
    global_metrics={"mean_subject_balanced_accuracy":float(np.mean(vals)),"subject_bootstrap_95_ci":bootstrap_ci(vals,BASE_SEED,CONFIG.get("bootstrap_iterations",10000)),"n_subjects":len(subject_rows),"n_folds_total":len(rows),"collapse_rate":float(np.mean([r["collapse_diagnostics"]["collapsed"] for r in rows])),"mean_training_seconds":float(np.mean([r["training_seconds"] for r in rows]))}
    return subject_rows,global_metrics
SUBJECT_ROWS,GLOBAL_METRICS=aggregate_results(FOLD_RESULTS)

In [ ]:
ref=Path(CONFIG["classical_reference_artifact"])
if ref.exists():
    if ref.suffix.lower()==".json":
        reference=pd.DataFrame(json.loads(ref.read_text(encoding="utf-8")))
    elif ref.suffix.lower()==".csv":
        reference=pd.read_csv(ref)
    else:
        raise ValueError(f"Unsupported classical reference format: {ref.suffix}")
    method=CONFIG.get("classical_reference_method")
    if method and "method" in reference.columns:
        reference=reference.loc[reference["method"]==method].copy()
    if reference.empty:
        raise ValueError(f"No classical reference rows found for method={method!r} in {ref}")
    normalize_subject_id=lambda value: f"sub-{int(str(value).replace('sub-','')):02d}"
    reference["subject_id"]=reference["subject_id"].map(normalize_subject_id)
    if reference["subject_id"].duplicated().any():
        raise ValueError(f"Classical reference must have one row per subject after method filtering: {ref}")
    ba_candidates=[c for c in ("subject_balanced_accuracy","balanced_accuracy","mean_balanced_accuracy") if c in reference.columns]
    if not ba_candidates:
        raise ValueError(f"No supported balanced-accuracy column in classical reference: {ref}")
    reference=reference[["subject_id",ba_candidates[0]]].rename(columns={ba_candidates[0]:"reference_balanced_accuracy"})
    current=pd.DataFrame(SUBJECT_ROWS); current["subject_id"]=current["subject_id"].map(normalize_subject_id)
    current_ids=set(current["subject_id"]); reference_ids=set(reference["subject_id"])
    if current_ids==reference_ids:
        merged=current.merge(reference,on="subject_id",validate="one_to_one")
        stat,p=wilcoxon(merged["balanced_accuracy"],merged["reference_balanced_accuracy"]); GLOBAL_METRICS["paired_wilcoxon_vs_confirmed_motor13"]={"statistic":float(stat),"p_value":float(p),"n":len(merged),"reference_path":str(ref),"reference_method":method}
    else:
        GLOBAL_METRICS["classical_reference_comparison_skipped"]={"reason":"configured cohort does not exactly match reference cohort","n_current":len(current_ids),"n_reference":len(reference_ids),"reference_path":str(ref),"reference_method":method}
GLOBAL_METRICS["holm_scope"]="Across six primary model runs; apply Holm after collecting their paired subject rows. No within-run p-value can perform cross-run correction."

## 6.2 Performance Visualizations

In [ ]:
plt.figure(figsize=(8,3)); plt.bar([r["subject_id"] for r in SUBJECT_ROWS],[100*r["balanced_accuracy"] for r in SUBJECT_ROWS]); plt.axhline(50,color="k",ls="--"); plt.xticks(rotation=90); plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"compact_mi_subject_performance.png",dpi=150); plt.close()

## 6.3 Experiment Summary

In [ ]:
print(json.dumps(GLOBAL_METRICS,indent=2))

## 6.4 Model Diagnostics
Parameter count, timing, seed ensemble, and collapse diagnostics are in fold artifacts.

## 6.5 Save Artifacts

In [ ]:
cv_results_path=ARTIFACT_DIR/"cv_results.json"; cv_results_path.write_text(json.dumps(FOLD_RESULTS,indent=2),encoding="utf-8")
subject_metrics_path=ARTIFACT_DIR/"subject_metrics.json"; subject_metrics_path.write_text(json.dumps(SUBJECT_ROWS,indent=2),encoding="utf-8")
global_metrics_path=ARTIFACT_DIR/"global_metrics.json"; global_metrics_path.write_text(json.dumps(GLOBAL_METRICS,indent=2),encoding="utf-8")
pd.DataFrame(FOLD_RESULTS).to_csv(ARTIFACT_DIR/"fold_results.csv",index=False); pd.DataFrame(SUBJECT_ROWS).to_csv(ARTIFACT_DIR/"subject_results.csv",index=False)
run_metadata={"run_id":RUN_ID,"artifact_dir":str(ARTIFACT_DIR),"experiment_name":CONFIG["experiment_name"],"config_note":CONFIG["config_note"],"subjects":SUBJECTS,"channel_names":CH_NAMES,"model_name":CONFIG.get("model_name"),"seed":BASE_SEED,"global_metrics":GLOBAL_METRICS,"artifacts":{p.name:str(p) for p in ARTIFACT_DIR.iterdir()}}
run_metadata_path=ARTIFACT_DIR/"run_metadata.json"; run_metadata_path.write_text(json.dumps(run_metadata,indent=2),encoding="utf-8")
print(f"CV results saved to:      {cv_results_path}"); print(f"Subject metrics saved to: {subject_metrics_path}"); print(f"Global metrics saved to:  {global_metrics_path}"); print(f"Run metadata saved to:    {run_metadata_path}"); print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass